In [6]:
import sys
print(sys.path)

['C:\\Users\\ASUS\\AppData\\Roaming\\uv\\python\\cpython-3.12.12-windows-x86_64-none\\python312.zip', 'C:\\Users\\ASUS\\AppData\\Roaming\\uv\\python\\cpython-3.12.12-windows-x86_64-none\\DLLs', 'C:\\Users\\ASUS\\AppData\\Roaming\\uv\\python\\cpython-3.12.12-windows-x86_64-none\\Lib', 'C:\\Users\\ASUS\\AppData\\Roaming\\uv\\python\\cpython-3.12.12-windows-x86_64-none', 'c:\\Adri\\Work\\Personal Projects\\llm_engineering\\.venv', '', 'c:\\Adri\\Work\\Personal Projects\\llm_engineering\\.venv\\Lib\\site-packages']


In [ ]:
# import sys
# import os
# sys.path.append(os.path.abspath("..../"))

In [5]:
import sys
print(sys.executable)

c:\Adri\Work\Personal Projects\llm_engineering\.venv\Scripts\python.exe


In [12]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
import time
from bs4 import BeautifulSoup

ModuleNotFoundError: No module named 'selenium'

In [ ]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


In [ ]:
# To give you a preview -- calling OpenAI with these messages is this easy. Any problems, head over to the Troubleshooting notebook.

message = "Hello, GPT! This is my first ever message to you! Hi!"

messages = [{"role": "user", "content": message}]


openai = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)
model = "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning:free"
# model = "deepseek/deepseek-v4-flash:free"

response = openai.chat.completions.create(model=model, messages=messages)
response.choices[0].message.content


In [ ]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model = model,
        messages = messages_for(website)
    )
    return response.choices[0].message.content

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [ ]:
display_summary("https://edwarddonner.com")

In [ ]:


# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.
"""

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

def fetch_js_website_contents(url, wait_time=3):
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    driver = webdriver.Chrome(ChromeDriverManager().install(), options=options)
    driver.get(url)
    time.sleep(wait_time)
    html = driver.page_source
    driver.quit()
    return html

def summarize_js_website(url, wait_time=3):
    website_html = fetch_js_website_contents(url, wait_time=wait_time)
    website_text = BeautifulSoup(website_html, "html.parser").get_text(separator="\n", strip=True)
    response = openai.chat.completions.create(
        model=model,
        messages=messages_for(website_text)
    )
    return response.choices[0].message.content

def display_js_summary(url, wait_time=3):
    summary = summarize_js_website(url, wait_time=wait_time)
    display(Markdown(summary))
    return summary